## Summary: Learning Outcomes Covered

### LO 1: Data & Preprocessing ✓
- **CLAHE**: Contrast Limited Adaptive Histogram Equalization for local contrast enhancement
- **Bilateral Filter**: Denoise while preserving sharp pothole edges
- Handles varied Indonesian road conditions (shadows, water reflections, lighting variations)

### LO 2: Feature Engineering & Descriptors ✓
- **Feature A**: Grayscale Intensity (identifies dark regions)
- **Feature B**: Local Variance (texture roughness - potholes > shadows)
- **Feature C**: Sobel Gradient (edge density at pothole boundaries)
- 3D feature vector for robust clustering

### LO 3: Smart Clustering ✓
- **K-Means with k=4**: Clusters features into 4 groups
- **Automated Selection**: Identifies pothole cluster by scoring:
  - Score = (1 - intensity) × variance
  - Dark + Textured = Potholes
  - Distinguishes from shadows, lane markings, cracks

### LO 4: Advanced Post-processing ✓
- **Morphological Closing**: Fills internal gaps (5×5 kernel)
- **Connected Component Analysis**: Analyzes individual components
- **Geometric Filtering**:
  - Area: Remove noise (< 20 px)
  - Solidity: Keep compact objects (> 0.5)
  - Aspect Ratio: Avoid elongated objects (< 3.0)

### LO 5: Competition Submission ✓
- **RLE Encoding**: Converts masks to run-length format
- **Dice Coefficient**: Evaluates accuracy on training set
- **Submission.csv**: Contains filename + RLE for all test images

---

## Key Design Decisions

1. **No Deep Learning**: Uses only OpenCV, NumPy, Scikit-Learn (as required)
2. **3D Feature Space**: Combines intensity, texture, and edges for better discrimination
3. **Automated Cluster Selection**: No manual threshold tuning needed
4. **Geometric Filtering**: Removes false positives (shadows, cracks, markings)
5. **PEP 8 & Type-Hinted**: Professional code quality for BINUS standards

In [1]:
# Generate Competition Submission

def generate_submission(
    pipeline: PotholeSegmentationPipeline,
    test_img_dir: Path,
    output_csv: Path
) -> None:
    """
    Generate submission.csv with RLE-encoded masks for all test images.
    """
    test_images = sorted(list(test_img_dir.glob('*')))
    results = []
    
    print(f"Processing {len(test_images)} test images...")
    
    for img_path in tqdm(test_images):
        try:
            image = cv2.imread(str(img_path))
            mask = pipeline.segment_image(image)
            rle = encode_rle(mask)
            
            results.append({
                'filename': img_path.name,
                'rle_mask': rle
            })
        except Exception as e:
            print(f"Error processing {img_path.name}: {e}")
            results.append({
                'filename': img_path.name,
                'rle_mask': ''
            })
    
    # Write to CSV
    with open(output_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['filename', 'rle_mask'])
        writer.writeheader()
        writer.writerows(results)
    
    print(f"\n{'='*60}")
    print(f"SUBMISSION GENERATED")
    print(f"{'='*60}")
    print(f"Output file: {output_csv}")
    print(f"Total images: {len(results)}")
    print(f"{'='*60}\n")

# Generate submission
output_csv = PROJECT_DIR / 'submission.csv'
generate_submission(pipeline, TEST_IMG_DIR, output_csv)

# Verify submission
if output_csv.exists():
    with open(output_csv, 'r') as f:
        lines = f.readlines()
    print(f"✓ Submission file created with {len(lines) - 1} entries")

NameError: name 'PotholeSegmentationPipeline' is not defined

In [ ]:
# Validation on Training Set

def validate_pipeline(pipeline: PotholeSegmentationPipeline, num_samples: int = 50) -> Tuple[float, List[float]]:
    """
    Validate pipeline on training set.
    Calculates average Dice Coefficient.
    """
    dice_scores = []
    
    train_images = sorted(list(TRAIN_IMG_DIR.glob('*')))[:num_samples]
    train_masks = sorted(list(TRAIN_MASK_DIR.glob('*')))[:num_samples]
    
    print(f"Validating on {len(train_images)} samples...")
    
    for img_path, mask_path in tqdm(zip(train_images, train_masks), total=len(train_images)):
        try:
            image = cv2.imread(str(img_path))
            mask = pipeline.segment_image(image)
            
            gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            dice = calculate_dice_coefficient(mask, gt_mask)
            dice_scores.append(dice)
        except Exception as e:
            print(f"Error: {img_path} - {e}")
            continue
    
    avg_dice = np.mean(dice_scores) if dice_scores else 0.0
    
    print(f"\n{'='*60}")
    print(f"VALIDATION RESULTS")
    print(f"{'='*60}")
    print(f"Average Dice Coefficient: {avg_dice:.4f}")
    print(f"Min Dice: {np.min(dice_scores):.4f}")
    print(f"Max Dice: {np.max(dice_scores):.4f}")
    print(f"Median Dice: {np.median(dice_scores):.4f}")
    print(f"Std Dev: {np.std(dice_scores):.4f}")
    print(f"{'='*60}\n")
    
    return avg_dice, dice_scores

# Run validation
avg_dice, dice_scores = validate_pipeline(pipeline, num_samples=50)

In [ ]:
# LO 5: RLE Encoding & Competition Submission

def encode_rle(mask: np.ndarray, pos_value: int = 255) -> str:
    """
    Run-Length Encoding for competition submission.
    
    Converts binary mask to RLE format as specified.
    """
    binary = (mask == pos_value).astype(np.uint8)
    pixels = binary.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)

def calculate_dice_coefficient(predicted: np.ndarray, ground_truth: np.ndarray) -> float:
    """
    Dice Coefficient = 2 * |X ∩ Y| / (|X| + |Y|)
    Ranges from 0 to 1, where 1 is perfect overlap.
    """
    predicted_binary = (predicted > 0).astype(np.uint8)
    gt_binary = (ground_truth > 0).astype(np.uint8)
    
    intersection = np.logical_and(predicted_binary, gt_binary).sum()
    dice = (2 * intersection) / (predicted_binary.sum() + gt_binary.sum() + 1e-8)
    
    return dice

# Test RLE encoding
test_rle = encode_rle(result_mask)
print(f"RLE encoded (first 100 chars): {test_rle[:100]}")
print(f"RLE length: {len(test_rle.split())}")

# Test Dice calculation
if list(TRAIN_MASK_DIR.glob('*')):
    test_mask_path = list(TRAIN_MASK_DIR.glob('*'))[0]
    gt_mask = cv2.imread(str(test_mask_path), cv2.IMREAD_GRAYSCALE)
    dice = calculate_dice_coefficient(result_mask, gt_mask)
    print(f"Dice Coefficient (single test): {dice:.4f}")

In [ ]:
# Complete Segmentation Pipeline

class PotholeSegmentationPipeline:
    """Complete unsupervised pothole segmentation pipeline."""
    
    def __init__(self, k_clusters: int = 4):
        self.k_clusters = k_clusters
    
    def segment_image(self, image: np.ndarray) -> np.ndarray:
        """
        Full pipeline: Preprocessing → Features → Clustering → Selection → Post-processing
        """
        # Extract features
        features = extract_features(image)
        
        # K-Means clustering
        labels, kmeans, scaler = cluster_features(features, self.k_clusters)
        
        # Automated cluster selection
        pothole_cluster, _ = select_pothole_cluster(labels, features, self.k_clusters)
        
        # Binary mask from selected cluster
        binary_mask = (labels == pothole_cluster).astype(np.uint8) * 255
        
        # Post-processing
        final_mask = post_process_mask(binary_mask)
        
        return final_mask

# Test full pipeline
pipeline = PotholeSegmentationPipeline(k_clusters=4)
result_mask = pipeline.segment_image(test_img)
print(f"Segmentation complete! Mask pixels: {result_mask.sum() // 255}")

In [ ]:
# LO 4: Advanced Post-processing with Geometric Filtering

def post_process_mask(mask: np.ndarray) -> np.ndarray:
    """
    Advanced post-processing pipeline:
    
    1. Morphological Closing: Fill internal gaps
       - Kernel: 5x5 ellipse
       - Connects nearby regions
    
    2. Connected Component Analysis with Geometric Filtering:
       - Area filtering: Remove very small noise (< 20 pixels)
       - Solidity filtering: ratio of area to convex hull area > 0.5
         * Potholes are compact (high solidity)
         * Shadows/cracks are elongated (low solidity)
       - Aspect Ratio filtering: Remove highly elongated objects
         * max(w,h) / min(w,h) < 3.0
         * Avoids lane markings and linear cracks
    """
    # Morphological closing: Fill internal gaps
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    # Connected components with statistics
    num_labels, labels_cc, stats, centroids = cv2.connectedComponentsWithStats(
        closed, connectivity=8
    )
    
    # Initialize output
    output = np.zeros_like(closed)
    
    # Filter components by geometric properties
    for i in range(1, num_labels):  # Skip background (0)
        area = stats[i, cv2.CC_STAT_AREA]
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        
        # Filter 1: Area (remove small noise)
        if area < 20:
            continue
        
        # Get component mask
        component_mask = (labels_cc == i).astype(np.uint8)
        
        # Filter 2: Solidity (area / convex hull area)
        contours, _ = cv2.findContours(
            component_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
        )
        if contours:
            hull = cv2.convexHull(contours[0])
            hull_area = cv2.contourArea(hull)
            if hull_area > 0:
                solidity = area / hull_area
            else:
                solidity = 0
        else:
            solidity = 0
        
        # Filter 3: Aspect ratio (avoid elongated objects)
        if w > 0 and h > 0:
            aspect_ratio = max(w, h) / min(w, h)
        else:
            aspect_ratio = 0
        
        # Keep components that pass all filters
        if solidity > 0.5 and aspect_ratio < 3.0:
            output[component_mask > 0] = 255
    
    return output

# Test post-processing
binary_mask = (labels == pothole_cluster).astype(np.uint8) * 255
final_mask = post_process_mask(binary_mask)

print(f"Raw mask - pixels: {binary_mask.sum() // 255}")
print(f"Final mask - pixels: {final_mask.sum() // 255}")
print(f"Reduction: {(1 - (final_mask.sum() / (binary_mask.sum() + 1e-8))) * 100:.1f}%")

In [ ]:
# LO 3: Smart K-Means Clustering with Automated Pothole Selection

def cluster_features(features: np.ndarray, k_clusters: int = 4) -> Tuple[np.ndarray, KMeans]:
    """
    Apply K-Means clustering on 3D feature space
    
    Args:
        features: (H, W, 3) feature tensor
        k_clusters: Number of clusters (default 4)
    
    Returns:
        labels: (H, W) cluster assignments
        kmeans: Fitted KMeans object
    """
    h, w = features.shape[:2]
    features_flat = features.reshape(-1, 3)
    
    # Standardize features for fair clustering
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features_flat)
    
    # K-Means with k=4 clusters
    kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(features_scaled)
    
    return labels.reshape(h, w), kmeans, scaler

def select_pothole_cluster(
    labels: np.ndarray, 
    features: np.ndarray, 
    k_clusters: int = 4
) -> int:
    """
    Automated Cluster Selector: Identify pothole cluster
    
    Logic:
    1. Potholes have LOWEST mean intensity (dark regions)
    2. Potholes have HIGHEST local variance (textured, rough)
    3. Calculate combined score: (1 - intensity) * variance
    4. Select cluster with highest score
    
    This distinguishes potholes from:
    - Shadows: Low intensity but low variance (smooth)
    - Road markings: High variance but high intensity
    - Cracks: Linear, will be filtered later
    """
    intensity = features[:, :, 0]
    variance = features[:, :, 1]
    
    cluster_scores: Dict[int, Dict] = {}
    
    for cluster_id in range(k_clusters):
        mask = labels == cluster_id
        
        # Skip very small clusters (likely noise)
        if mask.sum() < 100:
            continue
        
        mean_intensity = intensity[mask].mean()
        mean_variance = variance[mask].mean()
        
        # Combined score: prefer dark AND textured
        score = (1.0 - mean_intensity) * mean_variance
        
        cluster_scores[cluster_id] = {
            'score': score,
            'intensity': mean_intensity,
            'variance': mean_variance,
            'pixel_count': mask.sum()
        }
    
    if not cluster_scores:
        return 0
    
    # Find cluster with highest score
    pothole_cluster = max(cluster_scores, key=lambda x: cluster_scores[x]['score'])
    
    return pothole_cluster, cluster_scores

# Test clustering
labels, kmeans, scaler = cluster_features(features, k_clusters=4)
pothole_cluster, cluster_info = select_pothole_cluster(labels, features, k_clusters=4)

print(f"Selected pothole cluster: {pothole_cluster}")
print(f"\nCluster Analysis:")
for cid, info in cluster_info.items():
    print(f"  Cluster {cid}: score={info['score']:.4f}, intensity={info['intensity']:.4f}, "
          f"variance={info['variance']:.4f}, pixels={info['pixel_count']}")

In [ ]:
# LO 2: Feature Engineering - 3D Feature Vector

def extract_features(image: np.ndarray) -> np.ndarray:
    """
    Extract 3-dimensional feature vector for K-Means clustering:
    
    Feature A: Grayscale Intensity (normalized 0-1)
              - Potholes are darker regions
              
    Feature B: Local Variance (3x3 window)
              - Captured using mean and squared mean
              - Potholes have higher texture roughness than smooth shadows
              
    Feature C: Sobel Gradient Magnitude
              - Edge density at pothole boundaries
              - Potholes have distinct edges from surrounding road
    
    Returns shape (H, W, 3) for clustering
    """
    # Preprocess
    preprocessed = preprocess_image(image)
    
    # Feature A: Normalized grayscale intensity
    intensity = preprocessed.astype(np.float32) / 255.0
    
    # Feature B: Local variance (3x3 window)
    # Var(X) = E[X^2] - E[X]^2
    mean = cv2.blur(preprocessed.astype(np.float32), (3, 3))
    sqr_mean = cv2.blur(preprocessed.astype(np.float32) ** 2, (3, 3))
    variance = np.sqrt(np.maximum(sqr_mean - mean**2, 0)) / 255.0
    
    # Feature C: Sobel gradient magnitude (edge density)
    sobelx = cv2.Sobel(preprocessed, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(preprocessed, cv2.CV_32F, 0, 1, ksize=3)
    gradient = np.sqrt(sobelx**2 + sobely**2) / 255.0
    
    # Stack features into 3D tensor
    features = np.stack([intensity, variance, gradient], axis=-1)
    
    return features

# Test feature extraction
features = extract_features(test_img)
print(f"Feature tensor shape: {features.shape}")
print(f"Feature A (Intensity) - min: {features[:,:,0].min():.3f}, max: {features[:,:,0].max():.3f}")
print(f"Feature B (Variance) - min: {features[:,:,1].min():.3f}, max: {features[:,:,1].max():.3f}")
print(f"Feature C (Gradient) - min: {features[:,:,2].min():.3f}, max: {features[:,:,2].max():.3f}")

In [ ]:
# LO 1: Data Preprocessing with CLAHE & Bilateral Filter

def preprocess_image(image: np.ndarray) -> np.ndarray:
    """
    Preprocess image: CLAHE + Bilateral Filter
    
    - CLAHE: Local contrast enhancement for handling varied lighting conditions
    - Bilateral Filter: Denoise while preserving sharp pothole edges
    """
    # Convert BGR to LAB color space for better local contrast
    if len(image.shape) == 3 and image.shape[2] == 3:
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l_channel = lab[:, :, 0]
    else:
        l_channel = image if len(image.shape) == 2 else cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # CLAHE (Contrast Limited Adaptive Histogram Equalization)
    # Improves local contrast for shadows and water reflections
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l_channel.astype(np.uint8))
    
    # Bilateral filter: Denoise while preserving sharp edges
    # d=9: diameter of each pixel neighborhood
    # sigmaColor=75: Color space standard deviation
    # sigmaSpace=75: Coordinate space standard deviation
    denoised = cv2.bilateralFilter(l_enhanced, d=9, sigmaColor=75, sigmaSpace=75)
    
    return denoised

# Test preprocessing on first image
test_img = cv2.imread(str(list(TRAIN_IMG_DIR.glob('*'))[0]))
prep_result = preprocess_image(test_img)
print(f"Original shape: {test_img.shape}, Preprocessed shape: {prep_result.shape}")

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from typing import Tuple, List, Dict
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import csv
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

# Project directory
PROJECT_DIR = Path('c:/Users/dianr/OneDrive - Bina Nusantara/SEM 4/LAB AI Semester 4/LAB Computer Vision/Project_AoL')
TRAIN_IMG_DIR = PROJECT_DIR / 'dataset' / 'train' / 'images'
TRAIN_MASK_DIR = PROJECT_DIR / 'dataset' / 'train' / 'mask'
TEST_IMG_DIR = PROJECT_DIR / 'dataset' / 'test' / 'images'

print(f"Train images: {len(list(TRAIN_IMG_DIR.glob('*')))}")
print(f"Train masks: {len(list(TRAIN_MASK_DIR.glob('*')))}")
print(f"Test images: {len(list(TEST_IMG_DIR.glob('*')))}")

# Unsupervised Pothole Segmentation Pipeline
## ARA 7.0 Competition - Traditional CV (No Deep Learning)

**Objective:** Maximize Dice Coefficient using K-Means clustering with advanced feature engineering and geometric filtering.

**Learning Outcomes:**
1. Data preprocessing with CLAHE & Bilateral filtering
2. Feature engineering (intensity, variance, gradient)
3. Automated cluster selection
4. Geometric filtering (area, solidity, aspect ratio)
5. RLE encoding for submission